# Branch Performance Segmentation and Risk Classification

**Objective:** Build an interpretable classification system that identifies branches at risk of underperformance based on historical operational behavior. This system supports proactive operational intervention for regional managers.

**Business Context:** This notebook simulates an internal operations risk monitoring system used by regional managers to identify underperforming branches and prioritize interventions.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set style for plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

# Load data
daily_ops = pd.read_csv('../data/daily_operations.csv')
restaurant_master = pd.read_csv('../data/restaurant_master.csv')

# Convert date column
daily_ops['date'] = pd.to_datetime(daily_ops['date'])

print("Data loaded successfully")
print(f"Daily operations: {daily_ops.shape}")
print(f"Restaurant master: {restaurant_master.shape}")

## 1. Target Definition

Define branch performance status using transparent business rules:

**Underperforming (1):** A branch meets ANY of the following criteria:
- Net profit margin consistently in the bottom quartile across branches
- Negative profit for >30% of observed days
- High revenue volatility (top quartile) combined with weak profitability (bottom quartile profit margin)
- Cost-to-revenue ratio in top quartile across branches

**Healthy (0):** All other branches.

In [ ]:
# Calculate branch-level aggregates for target definition
branch_metrics = daily_ops.groupby('restaurant_id').agg({
    'revenue': ['mean', 'std'],
    'net_profit': ['mean', 'count'],
    'customers': 'mean',
    'food_cost': 'mean',
    'staff_cost': 'mean',
    'rent': 'mean',
    'operating_expenses': 'mean'
}).reset_index()

# Flatten column names
branch_metrics.columns = ['restaurant_id', 'avg_revenue', 'revenue_std', 'avg_profit', 'days_count', 'avg_customers', 
                          'avg_food_cost', 'avg_staff_cost', 'avg_rent', 'avg_op_exp']

# Calculate profit margin and cost-to-revenue ratio
branch_metrics['profit_margin'] = branch_metrics['avg_profit'] / branch_metrics['avg_revenue']
branch_metrics['cost_to_revenue'] = (branch_metrics['avg_food_cost'] + branch_metrics['avg_staff_cost'] + 
                                      branch_metrics['avg_rent'] + branch_metrics['avg_op_exp']) / branch_metrics['avg_revenue']

# Calculate negative profit proportion
negative_profit_days = daily_ops[daily_ops['net_profit'] < 0].groupby('restaurant_id').size().reset_index(name='negative_days')
branch_metrics = branch_metrics.merge(negative_profit_days, on='restaurant_id', how='left')
branch_metrics['negative_days'] = branch_metrics['negative_days'].fillna(0)
branch_metrics['negative_profit_prop'] = branch_metrics['negative_days'] / branch_metrics['days_count']

# Calculate quartiles for thresholds
profit_margin_q25 = branch_metrics['profit_margin'].quantile(0.25)
profit_margin_q75 = branch_metrics['profit_margin'].quantile(0.75)
revenue_vol_q75 = branch_metrics['revenue_std'].quantile(0.75)
cost_rev_q75 = branch_metrics['cost_to_revenue'].quantile(0.75)

# Define underperforming criteria
branch_metrics['underperforming'] = (
    (branch_metrics['profit_margin'] < profit_margin_q25) |  # Bottom quartile profit margin
    (branch_metrics['negative_profit_prop'] > 0.3) |  # >30% negative profit days
    ((branch_metrics['revenue_std'] > revenue_vol_q75) & (branch_metrics['profit_margin'] < profit_margin_q25)) |  # High vol + weak profit
    (branch_metrics['cost_to_revenue'] > cost_rev_q75)  # Top quartile cost-to-revenue
).astype(int)

# Merge with master data
branch_data = branch_metrics.merge(restaurant_master, on='restaurant_id', how='left')

print(f"Underperforming branches: {branch_data['underperforming'].sum()}")
print(f"Healthy branches: {len(branch_data) - branch_data['underperforming'].sum()}")
print(f"Class distribution: {branch_data['underperforming'].value_counts(normalize=True)}")

## 2. Feature Engineering

Aggregate features at branch level for modeling:

In [ ]:
# Financial Performance Features
features_df = branch_data[['restaurant_id', 'underperforming']].copy()
features_df['avg_revenue'] = branch_metrics['avg_revenue']
features_df['avg_net_profit'] = branch_metrics['avg_profit']
features_df['profit_margin'] = branch_metrics['profit_margin']
features_df['cost_to_revenue_ratio'] = branch_metrics['cost_to_revenue']

# Demand Features
features_df['avg_customer_volume'] = branch_metrics['avg_customers']
features_df['revenue_per_customer'] = branch_metrics['avg_revenue'] / branch_metrics['avg_customers']

# Operational Stability Features
features_df['profit_volatility'] = daily_ops.groupby('restaurant_id')['net_profit'].std().reset_index()['net_profit']
features_df['revenue_volatility'] = branch_metrics['revenue_std']
features_df['customer_volatility'] = daily_ops.groupby('restaurant_id')['customers'].std().reset_index()['customers']

# Fill NaN volatilities with 0 (for branches with single day)
features_df = features_df.fillna(0)

# Structural Features
features_df = features_df.merge(restaurant_master[['restaurant_id', 'location_tier', 'base_rating']], on='restaurant_id')

# Encode location tier
location_encoding = {'CBD': 3, 'Airport': 2, 'Highway': 1, 'Suburb': 0}
features_df['location_tier_encoded'] = features_df['location_tier'].map(location_encoding)

# Select final features
feature_cols = ['avg_revenue', 'avg_net_profit', 'profit_margin', 'cost_to_revenue_ratio',
                'avg_customer_volume', 'revenue_per_customer', 'profit_volatility', 
                'revenue_volatility', 'customer_volatility', 'location_tier_encoded', 'base_rating']

X = features_df[feature_cols]
y = features_df['underperforming']

print(f"Features shape: {X.shape}")
print(f"Feature columns: {feature_cols}")

## 3. Data Leakage Control

All features are computed from historical data only. No future information is used. Temporal integrity is preserved through aggregation at branch level.

## 4. Model Training

Train Logistic Regression for interpretability:

In [ ]:
# Train/test split (stratified by target)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Logistic Regression
model = LogisticRegression(random_state=42, class_weight='balanced')
model.fit(X_train_scaled, y_train)

# Predictions
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

print("Model trained successfully")

## 5. Model Evaluation

Evaluate from business risk perspective:

In [ ]:
# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.3f}")
print(f"Precision (Underperforming): {precision:.3f}")
print(f"Recall (Underperforming): {recall:.3f}")
print(f"F1-Score: {f1:.3f}")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

# Business interpretation
tn, fp, fn, tp = cm.ravel()
print(f"\nBusiness Risk Assessment:")
print(f"False Negatives (missed underperforming): {fn} - HIGH RISK")
print(f"False Positives (false alarms): {fp} - ACCEPTABLE")
print(f"True Positives (correctly identified): {tp}")
print(f"True Negatives (correctly healthy): {tn}")

## 6. Feature Importance & Explainability

In [ ]:
# Feature importance from coefficients
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': np.abs(model.coef_[0])
}).sort_values('importance', ascending=False)

print("Top Drivers of Underperformance:")
for idx, row in feature_importance.head(5).iterrows():
    print(f"- {row['feature']}: {row['importance']:.3f}")

# Business interpretation
print("\nBusiness Insights:")
print("• High cost-to-revenue ratio is the strongest predictor of risk")
print("• Low profit margins combined with high volatility indicate instability")
print("• Location tier and base rating influence performance expectations")

## 7. Risk Scoring System

In [ ]:
# Get risk scores for all branches
X_all_scaled = scaler.transform(X)
risk_scores = model.predict_proba(X_all_scaled)[:, 1]

# Classify risk levels
def classify_risk(score):
    if score >= 0.7:
        return 'High Risk'
    elif score >= 0.4:
        return 'Medium Risk'
    else:
        return 'Low Risk'

features_df['risk_score'] = risk_scores
features_df['risk_level'] = features_df['risk_score'].apply(classify_risk)
features_df['predicted_class'] = model.predict(X_all_scaled)

print("Risk scoring applied to all branches")

## 8. Output Deliverables

In [ ]:
# A. Ranked Risk Table
risk_table = features_df.merge(restaurant_master[['restaurant_id', 'restaurant_name', 'location_tier']], on='restaurant_id')
risk_table = risk_table[['restaurant_id', 'restaurant_name', 'location_tier', 'risk_score', 'risk_level', 
                         'predicted_class', 'underperforming', 'avg_revenue', 'profit_margin', 'cost_to_revenue_ratio']]
risk_table = risk_table.sort_values('risk_score', ascending=False).round(4)

print("=== RANKED RISK TABLE ===")
print(risk_table.head(10).to_string(index=False))

In [ ]:
# B. High-Risk Branch Report
high_risk = risk_table[risk_table['risk_level'] == 'High Risk']

print("\n=== HIGH-RISK BRANCH REPORT ===")
for idx, row in high_risk.iterrows():
    print(f"\nBranch: {row['restaurant_name']} (ID: {row['restaurant_id']})")
    print(f"Location: {row['location_tier']}")
    print(f"Risk Score: {row['risk_score']:.3f}")
    print(f"Key Issues:")
    if row['cost_to_revenue_ratio'] > branch_metrics['cost_to_revenue'].quantile(0.75):
        print(f"  - Cost-to-revenue ratio: {row['cost_to_revenue_ratio']:.3f} (above 75th percentile)")
    if row['profit_margin'] < branch_metrics['profit_margin'].quantile(0.25):
        print(f"  - Profit margin: {row['profit_margin']:.3f} (below 25th percentile)")
    print(f"Recommended Actions: Cost control measures, operational efficiency review")

## 9. Business Interpretation Layer

**Decision Support Summary:**

This risk monitoring system identifies branches likely to underperform, enabling proactive intervention. Key findings:

- **High-Risk Branches:** Require immediate attention for cost management and profitability improvement
- **Medium-Risk Branches:** Monitor closely for early warning signals
- **Primary Risk Drivers:** Cost-to-revenue ratio and profit margin stability

**Operational Recommendations:**
- Focus interventions on high-cost branches with volatile revenues
- Prioritize CBD locations with efficiency challenges
- Use this system monthly to track performance trends